In [5]:
import pandas as pd
import re 
import numpy as np 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

'Simple script for sorting movies alphabetically'

# Reading CSV files into dataframes
movies_df = pd.read_csv("movies.csv", low_memory = False)
movies_df['clean_title'] = movies_df['title'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True).str.lower().str.strip()

# Cleaning up movie titles that start with "The", "A", or "An" and moving them to front of title
movies_df['title'] = movies_df['title'].str.replace(
    r'^(.*),\s*(The|A|An)(\s+\(\d{4}\))?$', 
    r'\2 \1\3', 
    regex=True, 
    case=False
).str.strip()

# Reading rating CSV file, skip bad lines since an error was returned 
ratings_df = pd.read_csv("ratings.csv", dtype={'userId': int, 'movieId': int, 'rating': float}, on_bad_lines='skip')

# Sorting the movies alphabetically, ignore special characters / case
movies_df['sort_key'] = movies_df['title'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True).str.lower().str.strip()
movies_df = movies_df.sort_values(by = 'sort_key', ascending = True)
movies_df = movies_df.drop(columns=['sort_key']).reset_index(drop=True)

# Cleaning genre column by removing pipe separator
movies_df['genres_clean'] = movies_df['genres'].str.replace('|', ' ', regex = False)

# New column that combines title and cleaned genres
movies_df['features'] = movies_df['title'] + " " + movies_df['genres_clean']

# Creating a movies df with simpler format, to allow for compatibility with user input

tfidf = TfidfVectorizer(stop_words='english')

# Constructing TF-IDF matrix (rows = movies, columns = unique vocabulary words)
tfidf_matrix = tfidf.fit_transform(movies_df['features'])
print(f"Matrix shape: {tfidf_matrix.shape}")

# Use cosine similarity to find similarity between all movies based off of TF-IDF
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Cosine similarity matrix shape: {cosine_sim.shape}")

# Create a series mapping movie titles to index in dataframe for easier lookup
indices = pd.Series(movies_df.index, index=movies_df['clean_title']).drop_duplicates()


'''Here we create the main function, that takes a movie and returns a list of the top 10 most similar movies based off the cos similarity matrix established'''
def get_recommendations(title, cosine_sim = cosine_sim):

    title_clean = re.sub(r'[^a-zA-Z0-9\s]', '', title).lower().strip()
    # First check if there is an exact match for the title in the dataset
    if title_clean not in indices:
    # Check if we can find a partial match for the title in the dataset
        matching_rows = movies_df[movies_df['clean_title'].str.contains(title_clean, regex=False)]
        
        if not matching_rows.empty:
            # Automatically grab the clean title of the first partial match found
            title_clean = matching_rows['clean_title'].iloc[0]
            matched_original_title = matching_rows['title'].iloc[0]
            print(f"(Matched '{title}' to '{matched_original_title}')")
        else:
            return f"Movie {title} not found in the dataset. Please check for typos or try a different movie."
    # get index of movie that matches title
    idx = indices[title_clean]

    # get pairwise similarity scores of all movies with given movie 
    similarity_scores = list(enumerate(cosine_sim[idx]))

    # sort from most simliar to least and choose top 10
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    similarity_scores = similarity_scores[1:11]

    # get movie indices and format as list of titles 
    movie_indices = [i[0] for i in similarity_scores]
    return movies_df['title'].iloc[movie_indices].tolist()


# Input system for the user to enter a movie title and get recommendations
print("Movie recommendation system. Type quit or exit to leave the program at any time. \nPlease enter a movie title to get recommendations :")
while True:
    user_input = input("Movie Title :")
    if user_input.lower() in ['quit', 'exit']:
        print("Ending program.")
        break
    recommendations = get_recommendations(user_input)

    # Use isinstance to prevent errors if input is not in dataset
    if isinstance(user_input, str) and isinstance(recommendations,list):
        for i, movie in enumerate(recommendations, 1):
            print(f"  {i}. {movie}")
        print("-" * 40)
    else:
        print(recommendations)
        print("-" * 40)

#### Next step: Ignore sequels/prequels and 

Matrix shape: (9742, 9060)
Cosine similarity matrix shape: (9742, 9742)
Movie recommendation system. Type quit or exit to leave the program at any time. 
Please enter a movie title to get recommendations :
(Matched 'toy story' to 'Toy Story (1995)')
  1. Toy Story 2 (1999)
  2. Toy Story 3 (2010)
  3. The Toy (1982)
  4. We're Back! A Dinosaur's Story (1993)
  5. Now and Then (1995)
  6. Toy Soldiers (1991)
  7. The NeverEnding Story (1984)
  8. The Wild (2006)
  9. Jumanji (1995)
  10. Balto (1995)
----------------------------------------
(Matched 'the dark knight' to 'Batman: The Dark Knight Returns, Part 1 (2012)')
  1. Batman: The Dark Knight Returns, Part 2 (2013)
  2. Batman Returns (1992)
  3. The Dark Knight (2008)
  4. Batman: Gotham Knight (2008)
  5. The Dark Knight Rises (2012)
  6. First Knight (1995)
  7. Alone in the Dark (2005)
  8. Superman Returns (2006)
  9. The Batman/Superman Movie (1998)
  10. Batman: Year One (2011)
----------------------------------------
Movie 